In [1]:
import os

In [3]:
os.chdir('../')

In [2]:
%pwd

'e:\\MLOPs Deep Learning Project\\research'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file:Path
    unzip_dir: Path

In [5]:
from cnnClassifier.constants import * #CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])
        print(params_file_path)
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
        root_dir=Path(config.root_dir),
        source_URL=config.source_URL,
        local_data_file=Path(config.local_data_file),
        unzip_dir=Path(config.unzip_dir)
            )

        return data_ingestion_config

In [ ]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size
from pathlib import Path

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        try:
            if not os.path.exists(self.config.local_data_file):
                logger.info(f"Downloading file from {self.config.source_URL} to {self.config.local_data_file}")
                os.makedirs("artifacts/data_ingestion", exist_ok=True)

                # Use source_URL from config instead of undefined dataset_url
                file_id = self.config.source_URL.split("/")[-2]
                prefix = 'https://drive.google.com/file/d/'
                gdown.download(prefix + file_id+"/view?usp=sharing", output=str(self.config.local_data_file))
            else:
                logger.info(f"File already exists at {self.config.local_data_file}")
        except Exception as e:
            logger.exception(f"An error occurred during file download: {e}")

    def unzip_and_get_size(self):
        try:
            os.makedirs(self.config.unzip_dir, exist_ok=True)
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(self.config.unzip_dir)

            data_size = get_size(self.config.unzip_dir)
            logger.info(f"Unzipped data size: {data_size}")
        except Exception as e:
            logger.exception(f"An error occurred during unzip: {e}")


In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.unzip_and_get_size()
except Exception as e:
    logger.exception(f"An error occurred in the main execution block: {e}")    

[2026-05-17 18:17:00,942 : INFO : common : yaml file: config\config.yaml loaded successfully]
[2026-05-17 18:17:00,949 : INFO : common : yaml file: E:\MLOPs Deep Learning Project\params.yaml loaded successfully]
[2026-05-17 18:17:00,957 : INFO : common : created directory at: artifacts]
E:\MLOPs Deep Learning Project\params.yaml
[2026-05-17 18:17:00,962 : INFO : common : created directory at: artifacts/data_ingestion]
[2026-05-17 18:17:00,966 : INFO : 2640554227 : Downloading file from https://drive.google.com/file/d/1kH263O1V1CbrGufLQdRptPXSYQ3a2AHM/view?usp=sharing to artifacts\data_ingestion\chest_scan.zip]


Downloading...
From: https://drive.google.com/uc?id=1kH263O1V1CbrGufLQdRptPXSYQ3a2AHM
To: e:\MLOPs Deep Learning Project\artifacts\data_ingestion\chest_scan.zip
100%|██████████| 23.2M/23.2M [00:04<00:00, 5.34MB/s]


[2026-05-17 18:17:19,541 : INFO : 2640554227 : Unzipped data size: ~ 0 KB]
